In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# تمثيل أجهزة الحوسبة الكمومية للمترجم


<details>
<summary><b>إصدارات الحزم</b></summary>

تم تطوير الكود في هذه الصفحة باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو أحدث منها.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
لتحويل دائرة مجردة إلى دائرة ISA قادرة على التشغيل على وحدة معالجة كمومية (QPU) محددة، يحتاج المترجم إلى معلومات معينة عن تلك الوحدة. هذه المعلومات موجودة في مكانين: كائن `BackendV2` (أو النسخة القديمة `BackendV1`) الذي تخطط لإرسال المهام إليه، وخاصية `Target` الخاصة بالـ backend.

- يحتوي [`Target`](../api/qiskit/qiskit.transpiler.Target) على جميع القيود ذات الصلة بالجهاز، مثل بوابات الأساس الأصلية، وترابط الكيوبتات، ومعلومات النبضات أو التوقيت.
- يمتلك [`Backend`](../api/qiskit/qiskit.providers.BackendV2) `Target` افتراضيًا، ويحتوي على معلومات إضافية — مثل [`InstructionScheduleMap`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.pulse.InstructionScheduleMap)، ويوفر الواجهة لإرسال مهام الدوائر الكمومية.

يمكنك أيضًا تقديم المعلومات صراحةً للمترجم، مثلًا إذا كان لديك حالة استخدام خاصة، أو إذا كنت تعتقد أن هذه المعلومات ستساعد المترجم في توليد دائرة أكثر تحسينًا.

تعتمد دقة المترجم في إنتاج الدائرة الأنسب لجهاز معين على قدر المعلومات التي يملكها `Target` أو `Backend` عن قيوده.

> **Note:** بما أن كثيرًا من خوارزميات الترجمة الأساسية عشوائية الطابع، لا يوجد ضمان بالعثور على دائرة أفضل.

تعرض هذه الصفحة عدة أمثلة على تمرير معلومات QPU إلى المترجم. تستخدم هذه الأمثلة الـ target من الـ backend الوهمي [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke).

<span id="default-config"></span>
## الإعداد الافتراضي
أبسط طريقة لاستخدام المترجم هي تقديم جميع معلومات QPU عبر توفير `Backend` أو `Target`. لفهم كيفية عمل المترجم بشكل أفضل، شيّد دائرة وترجمها بمعلومات مختلفة على النحو التالي.

استورد المكتبات الضرورية وأنشئ مثيل QPU:
لتحويل دائرة مجردة إلى دائرة ISA قادرة على التشغيل على معالج محدد، يحتاج المترجم إلى معلومات معينة عن ذلك المعالج. عادةً ما تكون هذه المعلومات مخزّنة في [`Backend`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.Backend#backend) أو [`Target`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Target#target) المُمرَّر إلى المترجم، ولا يلزم تقديم أي معلومات إضافية. غير أنه يمكنك أيضًا تقديم المعلومات صراحةً للمترجم، مثلًا إذا كان لديك حالة استخدام خاصة، أو إذا كنت تعتقد أن هذه المعلومات ستساعده في توليد دائرة أكثر تحسينًا.

يعرض هذا الموضوع عدة أمثلة على تمرير المعلومات إلى المترجم. تستخدم هذه الأمثلة الـ target من الـ backend الوهمي [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke).

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
target = backend.target

تستخدم الدائرة في المثال مثيلًا من [`efficient_su2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) من مكتبة دوائر Qiskit.

In [2]:
from qiskit.circuit.library import efficient_su2

qc = efficient_su2(12, entanglement="circular", reps=1)

qc.draw("mpl")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg)

يستخدم هذا المثال الإعدادات الافتراضية للترجمة إلى `target` الخاص بـ `backend`، الذي يوفر كافة المعلومات اللازمة لتحويل الدائرة إلى دائرة قابلة للتشغيل على الـ backend.

In [3]:
from qiskit.transpiler import generate_preset_pass_manager

pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=target, seed_transpiler=12345
)
qc_t_target = pass_manager.run(qc)
qc_t_target.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg)

يُستخدم هذا المثال في الأقسام اللاحقة من هذا الموضوع لتوضيح أن خريطة الاقتران وبوابات الأساس هما المعلومتان الجوهريتان اللتان يجب تمريرهما إلى المترجم لبناء دائرة مثلى. يستطيع QPU عادةً اختيار الإعدادات الافتراضية للمعلومات الأخرى التي لم يتم تمريرها، كالتوقيت والجدولة.

## خريطة الاقتران

خريطة الاقتران هي رسم بياني يُظهر الكيوبتات المترابطة وبالتالي الأزواج التي تعمل بينها البوابات ثنائية الكيوبت. أحيانًا يكون هذا الرسم موجهًا، بمعنى أن البوابات ثنائية الكيوبت لا تعمل إلا في اتجاه واحد. لكن المترجم يستطيع دائمًا عكس اتجاه البوابة عن طريق إضافة بوابات أحادية الكيوبت. يمكن تمثيل أي دائرة كمومية مجردة على هذا الرسم البياني، حتى لو كان الترابط محدودًا، وذلك بإدخال بوابات SWAP لنقل المعلومات الكمومية.

تُسمى الكيوبتات في دوائرنا المجردة _الكيوبتات الافتراضية_، بينما تُسمى تلك الموجودة في خريطة الاقتران _الكيوبتات الفيزيائية_. يوفر المترجم تعيينًا بين الكيوبتات الافتراضية والفيزيائية. تُنفَّذ هذه العملية في مرحلة _التخطيط_، وهي إحدى أولى مراحل الترجمة.

> **Note:** رغم أن مرحلة التوجيه مرتبطة ارتباطًا وثيقًا بمرحلة _التخطيط_ — التي تختار الكيوبتات الفعلية — يتعامل هذا الموضوع معهما كمرحلتين منفصلتين للتبسيط. يُسمى الجمع بين التوجيه والتخطيط _تعيين الكيوبت_.  اطّلع على مزيد من التفاصيل حول هذه المراحل في موضوع [مراحل المترجم](transpiler-stages).

مرّر وسيطة الكلمة المفتاحية `coupling_map` لمشاهدة تأثيرها على المترجم:

In [4]:
coupling_map = target.build_coupling_map()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv0 = pass_manager.run(qc)
qc_t_cm_lv0.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg)

كما هو موضح أعلاه، تم إدراج عدة بوابات SWAP (تتكون كل منها من ثلاث بوابات CX)، مما سيتسبب في أخطاء كثيرة على الأجهزة الحالية. لمعرفة الكيوبتات المختارة على طوبولوجيا الكيوبتات الفعلية، استخدم `plot_circuit_layout` من أدوات التصور في Qiskit:

In [5]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv0, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg)

هذا يُظهر أن الكيوبتات الافتراضية 0-11 تم تعيينها مباشرةً إلى سلسلة الكيوبتات الفيزيائية 0-11. لنعد إلى الإعداد الافتراضي (`optimization_level=1`)، الذي يستخدم `VF2Layout` إذا كان أي توجيه مطلوبًا.

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv1 = pass_manager.run(qc)
qc_t_cm_lv1.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg)

الآن لم يتم إدراج أي بوابات SWAP، والكيوبتات الفيزيائية المختارة هي نفسها عند استخدام كلاس `target`.

In [7]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv1, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg)

أصبح التخطيط الآن على شكل حلقة. ولأن هذا التخطيط يحترم ترابط الدائرة، لا توجد بوابات SWAP، مما يوفر دائرة أفضل بكثير للتنفيذ.

## بوابات الأساس

كل حاسوب كمومي يدعم مجموعة محدودة من التعليمات تُسمى _بوابات الأساس_. يجب ترجمة كل بوابة في الدائرة إلى عناصر هذه المجموعة. ينبغي أن تتكون هذه المجموعة من بوابات أحادية وثنائية الكيوبت توفر مجموعة بوابات شاملة، بمعنى أنه يمكن تحليل أي عملية كمومية إلى تلك البوابات. يتولى هذا [BasisTranslator](../api/qiskit/qiskit.transpiler.passes.BasisTranslator)، ويمكن تحديد بوابات الأساس كوسيطة كلمة مفتاحية للمترجم لتقديم هذه المعلومات.

In [8]:
basis_gates = list(target.operation_names)
print(basis_gates)

['sx', 'switch_case', 'x', 'if_else', 'measure', 'for_loop', 'delay', 'ecr', 'id', 'reset', 'rz']


The default single-qubit gates on _ibm_sherbrooke_ are `rz`, `x`, and `sx`, and the default two-qubit gate is `ecr` (echoed cross-resonance). CX gates are constructed from `ecr` gates, so on some QPUs `ecr` is specified as the two-qubit basis gate, while on others `cx` is the default. The `ecr` gate is the _entangling_ part of the `cx` gate. In addition to the control gates, there are also `delay` and `measurement` instructions.


<Admonition type="note">
    QPUs have default basis gates, but you can choose whatever gates you want, as long as you provide the instruction or add pulse gates (see [Create transpiler passes.](custom-transpiler-pass)) The default basis gates are those that calibrations have been done for on the QPU, so no further instruction/pulse gates need to be provided. For example, on some QPUs `cx` is the default two-qubit gate and `ecr` on others. See the list of possible [native gates and operations](/docs/guides/qpu-information#native-gates) for more details.
</Admonition>

In [9]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    coupling_map=coupling_map,
    basis_gates=basis_gates,
    seed_transpiler=12345,
)
qc_t_cm_bg = pass_manager.run(qc)
qc_t_cm_bg.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg" alt="Output of the previous code cell" />

البوابات الافتراضية أحادية الكيوبت على _ibm_sherbrooke_ هي `rz` و`x` و`sx`، والبوابة الافتراضية ثنائية الكيوبت هي `ecr` (الرنين المتقاطع المتردد). تُبنى بوابات CX من بوابات `ecr`، لذا على بعض وحدات QPU يُحدَّد `ecr` كبوابة أساس ثنائية الكيوبت، بينما `cx` هي الافتراضية على أخرى. بوابة `ecr` هي الجزء _المتشابك_ من بوابة `cx`. إضافةً إلى بوابات التحكم، توجد أيضًا تعليمات `delay` و`measurement`.

> **Note:** للـ QPUs بوابات أساس افتراضية، لكن يمكنك اختيار أي بوابات تريدها، طالما قدّمت التعليمة أو أضفت بوابات نبضية (انظر [إنشاء تمريرات المترجم.](custom-transpiler-pass)) بوابات الأساس الافتراضية هي تلك التي أُجريت لها معايرات على QPU، لذا لا حاجة لتقديم تعليمات/بوابات نبضية إضافية. فمثلًا، `cx` هي البوابة الافتراضية ثنائية الكيوبت على بعض وحدات QPU و`ecr` على أخرى. اطّلع على قائمة [البوابات والعمليات الأصلية المحتملة](/guides/qpu-information#native-gates) لمزيد من التفاصيل.

In [10]:
target["ecr"][(1, 0)]

InstructionProperties(duration=5.333333333333332e-07, error=0.007494257741828603)

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg)

لاحظ أن كائنات `CXGate` قد تم تحليلها إلى بوابات `ecr` وبوابات الأساس أحادية الكيوبت.
## معدلات أخطاء الجهاز
يمكن أن تحتوي كلاس `Target` على معلومات عن معدلات الأخطاء للعمليات على الجهاز.
فمثلًا، يسترد الكود التالي خصائص بوابة الرنين المتقاطع المتردد (ECR) بين الكيوبت 1 والكيوبت 0 (لاحظ أن بوابة ECR موجهة):

In [11]:
from qiskit.transpiler import Target
from qiskit.circuit.controlflow import IfElseOp, SwitchCaseOp, ForLoopOp

err_targ = Target.from_configuration(
    basis_gates=basis_gates,
    coupling_map=coupling_map,
    num_qubits=target.num_qubits,
    custom_name_mapping={
        "if_else": IfElseOp,
        "switch_case": SwitchCaseOp,
        "for_loop": ForLoopOp,
    },
)

for i, (op, qargs) in enumerate(target.instructions):
    if op.name in basis_gates:
        err_targ[op.name][qargs] = target.instruction_properties(i)

Transpile with our new target `err_targ` as the target:

In [12]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=err_targ, seed_transpiler=12345
)
qc_t_cm_bg_et = pass_manager.run(qc)
qc_t_cm_bg_et.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/f1e270c4-e2cc-487e-a050-4180bc321b0b-0.svg" alt="Output of the previous code cell" />

يعرض الناتج مدة البوابة (بالثواني) ومعدل خطئها. لكشف معلومات الأخطاء للمترجم، ابنِ نموذج target مع `basis_gates` و`coupling_map` من الأعلى واملأه بقيم الأخطاء من الـ backend `FakeSherbrooke`.